# scikit-learn だけで完結する探索

**出典**　機械学習 4 分野の教科書 — 第 5 章 1-D 探索の自動化とモデル選定 ／ 04 ライブラリに任せる探索——Grid・Random・Halving

**このノートのパス**　`ml-code/05_hpo/04_sklearn_search.ipynb`

**このノートで確かめること**
**前処理を必ず Pipeline に入れる**書き方と、`RandomizedSearchCV` / `HalvingRandomSearchCV`。
**学習率のような桁で効く量は `loguniform` で引く**ことを確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `lightgbm scikit-learn scipy numpy` |
| 実行時間の目安 | 約 8〜15 分（CPU コア数による） |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q lightgbm scikit-learn scipy numpy

In [3]:
import time, math, random
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier

SEED = 42
X, y = make_classification(
    n_samples=300_000, n_features=30, n_informative=15, n_redundant=8,
    weights=[0.95, 0.05], class_sep=1.3, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

# 探索は時間がかかるのでサブサンプルで回す（増やせば時間も増える）
N_SUB = 8_000
X_s, y_s = X_tr[:N_SUB], y_tr[:N_SUB]
print("探索に使う件数:", X_s.shape)

探索に使う件数: (8000, 30)


In [4]:
from sklearn.experimental import enable_halving_search_cv   # noqa: F401  ← 必須
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, HalvingRandomSearchCV, StratifiedKFold)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform, randint

# 前処理は必ず Pipeline に入れる。CV の外でスケーリングすると検証データの
# 平均・分散が訓練側に漏れる（＝リーク）。Pipeline なら fold ごとに fit される
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LGBMClassifier(class_weight="balanced", random_state=SEED,
                           subsample_freq=1, n_jobs=1, verbosity=-1)),
])

space = {
    "clf__n_estimators":  randint(150, 400),
    "clf__num_leaves":    randint(15, 120),
    "clf__learning_rate": loguniform(1e-2, 3e-1),   # 学習率は対数一様で引く
    "clf__subsample":     [0.6, 0.7, 0.8, 0.9, 1.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)  # 書籍では n_splits=5。CPU で試せるよう小さくしてある

## ① ランダム探索 — 予算は `n_iter` だけで決まる

In [5]:
search = RandomizedSearchCV(
    pipe, space,
    error_score="raise",           # 評価失敗を NaN に置き換えて続行しない
    n_iter=10,                      # ← 予算はここだけで決まる（書籍では 60）
    scoring="average_precision",    # PR-AUC。不均衡では ROC-AUC より素直
    cv=cv, n_jobs=-1, random_state=SEED, refit=True,
)
search.fit(X_s, y_s)                # テストデータ X_te には触らない
assert np.isfinite(search.cv_results_["mean_test_score"]).all()
print(round(search.best_score_, 5))
print(search.best_params_)

0.89276
{'clf__learning_rate': np.float64(0.08007296015034579), 'clf__n_estimators': 385, 'clf__num_leaves': 103, 'clf__subsample': 0.6}


## ② 逐次半減 — 候補を少ない資源で広く試し、生き残りに資源を増やす

In [6]:
hsearch = HalvingRandomSearchCV(
    pipe, space,
    min_resources="exhaust",        # 少数クラスが消えるほど小さい初期件数を避ける
    error_score="raise",
    n_candidates=9,                 # 最初に広く撒く（書籍では 200）
    factor=3,                       # 毎ラウンド 1/3 に絞り、資源を 3 倍にする
    resource="n_samples",           # 資源＝学習に使う件数
    scoring="average_precision", cv=cv, n_jobs=-1, random_state=SEED,
)
hsearch.fit(X_s, y_s)
assert np.isfinite(hsearch.cv_results_["mean_test_score"]).all()
print(round(hsearch.best_score_, 5))
print(hsearch.best_params_)

import pandas as pd
r = pd.DataFrame(hsearch.cv_results_)
print(r.groupby("iter").agg(候補数=("params", "size"),
                            使用件数=("n_resources", "max")).to_string())

0.89459
{'clf__learning_rate': np.float64(0.08007296015034579), 'clf__n_estimators': 385, 'clf__num_leaves': 103, 'clf__subsample': 0.6}
      候補数  使用件数
iter           
0       9   888
1       3  2664
2       1  7992


## ③ 対数一様（loguniform）で引く理由を、実際に引いて確かめる

In [7]:
from scipy.stats import uniform

rng = np.random.default_rng(0)
lin = uniform(0.01, 0.29).rvs(10_000, random_state=0)
log = loguniform(1e-2, 3e-1).rvs(10_000, random_state=0)

for name, v in [("uniform", lin), ("loguniform", log)]:
    low = (v < 0.03).mean()
    print(f"{name:>11}: 0.01〜0.03 の範囲に入った割合 = {low:.1%}")

    uniform: 0.01〜0.03 の範囲に入った割合 = 7.3%
 loguniform: 0.01〜0.03 の範囲に入った割合 = 32.9%


## 読み取りどころ

- **`enable_halving_search_cv` の import は必須。**書かないと `HalvingRandomSearchCV` が見つからない（実験的機能のため）。
- **`<推定器名>__<パラメータ名>`** で Pipeline の中を指す。`clf__num_leaves` のように。
- **`uniform(0.01, 0.3)` で引くと、0.01〜0.03 の領域がほとんど試されない。**上のセルで実測すると 1 割未満になる。`loguniform` なら 3 割強。
  **0.01 と 0.02 の差は、0.2 と 0.21 の差よりずっと大きい**のに、である。

**scikit-learn 公式の一文**——ランダム探索は
*"A budget can be chosen independent of the number of parameters and possible values. Adding parameters that do not influence the performance does not decrease efficiency."*
効かないパラメータを足しても効率が落ちない。格子探索ではこうはいかない。